In [ ]:

import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:

# Step 2: load only needed columns, dayfirst parsing
df = pd.read_csv(
    "your_data.csv",
    usecols=["trolleyId", "geozoneId", "createdOn", "status"],
    parse_dates=["createdOn"],
    dayfirst=True
)

df = df[(df["status"] == 1) & (df["geozoneId"] != 0)].copy()
df = df[["trolleyId", "geozoneId", "createdOn"]]


In [ ]:

# Step 3: dedup - one record per trolley/zone/hour
df = df.sort_values(["trolleyId", "geozoneId", "createdOn"])
df["date"] = df["createdOn"].dt.date
df["hour"] = df["createdOn"].dt.hour

df = df.drop_duplicates(subset=["trolleyId", "geozoneId", "date", "hour"])


In [ ]:

# Step 4: aggregate -> unique trolley count per zone/date/hour
agg = df.groupby(["date", "hour", "geozoneId"])["trolleyId"].nunique().reset_index()
agg.columns = ["date", "hour", "geozoneId", "trolley_count"]

# Build complete grid (date x hour x geozoneId) so missing slots become 0
all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D").date
all_hours = range(24)
all_zones = sorted(df["geozoneId"].unique())

grid = pd.MultiIndex.from_product(
    [all_dates, all_hours, all_zones],
    names=["date", "hour", "geozoneId"]
).to_frame(index=False)

full = grid.merge(agg, on=["date", "hour", "geozoneId"], how="left")
full["trolley_count"] = full["trolley_count"].fillna(0).astype(int)

full["datetime"] = pd.to_datetime(full["date"]) + pd.to_timedelta(full["hour"], unit="h")
full = full.sort_values(["geozoneId", "datetime"]).reset_index(drop=True)


In [ ]:

# Step 5: date-derivable features only
full["month"] = full["datetime"].dt.month
full["day_of_week"] = full["datetime"].dt.dayofweek
full["is_weekend"] = full["day_of_week"].isin([5, 6]).astype(int)
full["is_month_start"] = full["datetime"].dt.is_month_start.astype(int)
full["is_month_end"] = full["datetime"].dt.is_month_end.astype(int)

# peak hour flags
full["is_morning_peak"] = full["hour"].between(7, 9).astype(int)
full["is_evening_peak"] = full["hour"].between(17, 19).astype(int)

# cyclical encodings
full["hour_sin"] = np.sin(2 * np.pi * full["hour"] / 24)
full["hour_cos"] = np.cos(2 * np.pi * full["hour"] / 24)
full["month_sin"] = np.sin(2 * np.pi * full["month"] / 12)
full["month_cos"] = np.cos(2 * np.pi * full["month"] / 12)
full["dow_sin"] = np.sin(2 * np.pi * full["day_of_week"] / 7)
full["dow_cos"] = np.cos(2 * np.pi * full["day_of_week"] / 7)


In [ ]:

# Malaysia public holidays - date-derivable
my_holidays = holidays.Malaysia(years=range(full["datetime"].dt.year.min(), full["datetime"].dt.year.max() + 2))
holiday_dates = set(my_holidays.keys())

full["is_holiday"] = full["date"].apply(lambda d: 1 if d in holiday_dates else 0)
full["is_day_before_holiday"] = full["date"].apply(lambda d: 1 if (d + pd.Timedelta(days=1)) in holiday_dates else 0)
full["is_day_after_holiday"] = full["date"].apply(lambda d: 1 if (d - pd.Timedelta(days=1)) in holiday_dates else 0)


In [ ]:

# Step 6: time-based split (no shuffle)
full = full.sort_values("datetime").reset_index(drop=True)

full["day"] = full["datetime"].dt.day      # date-of-month (1-31)
full["month_num"] = full["datetime"].dt.month
full["year"] = full["datetime"].dt.year

cutoff = full["datetime"].max() - pd.Timedelta(days=14)
train = full[full["datetime"] <= cutoff]
test  = full[full["datetime"] > cutoff]

feature_cols = [
    "geozoneId", "hour", "day_of_week", "is_weekend",
    "is_month_start", "is_month_end",
    "is_morning_peak", "is_evening_peak",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "is_holiday", "is_day_before_holiday", "is_day_after_holiday"
]
target_col = "trolley_count"

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]


In [ ]:

model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
preds = np.clip(preds, 0, None)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")


In [ ]:

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)


In [ ]:

# ---- INFERENCE: given only a date, predict trolley_count for every (hour, geozoneId) ----

def build_features(dt, geozoneId):
    hour = dt.hour
    day_of_week = dt.dayofweek
    month = dt.month
    d = dt.date()

    is_holiday = 1 if d in holiday_dates else 0
    is_day_before_holiday = 1 if (d + pd.Timedelta(days=1)) in holiday_dates else 0
    is_day_after_holiday  = 1 if (d - pd.Timedelta(days=1)) in holiday_dates else 0

    return {
        "geozoneId": geozoneId,
        "hour": hour,
        "day_of_week": day_of_week,
        "is_weekend": int(day_of_week in (5, 6)),
        "is_month_start": int(dt.is_month_start),
        "is_month_end": int(dt.is_month_end),
        "is_morning_peak": int(7 <= hour <= 9),
        "is_evening_peak": int(17 <= hour <= 19),
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
        "dow_sin": np.sin(2 * np.pi * day_of_week / 7),
        "dow_cos": np.cos(2 * np.pi * day_of_week / 7),
        "is_holiday": is_holiday,
        "is_day_before_holiday": is_day_before_holiday,
        "is_day_after_holiday": is_day_after_holiday,
    }


In [ ]:

def predict_for_date(target_date_str):
    target_date = pd.to_datetime(target_date_str).date()
    rows = []

    for zone in all_zones:
        for hour in range(24):
            dt = pd.Timestamp(target_date) + pd.Timedelta(hours=hour)
            rows.append(build_features(dt, zone))

    X = pd.DataFrame(rows)[feature_cols]
    preds = model.predict(X)
    preds = np.clip(np.round(preds), 0, None).astype(int)

    out = pd.DataFrame(rows)[["geozoneId", "hour"]]
    out["date"] = target_date
    out["predicted_trolley_count"] = preds
    return out[["date", "hour", "geozoneId", "predicted_trolley_count"]]


In [ ]:

# Example usage
result_df = predict_for_date("2026-03-08")
result_df.head(30)


In [ ]:

# Pivot for a readable view: rows = hour, columns = geozoneId
pivot = result_df.pivot(index="hour", columns="geozoneId", values="predicted_trolley_count")
pivot
